# Run the whole pipeline on Kaggle
Settings: **Accelerator = GPU**, **Internet = On**, add input dataset *Chest X-Ray Images (Pneumonia)* (Paul Mooney). Then **Run All**. Training takes about 10 minutes.

In [ ]:
import os
for d in ('src','app'):
    os.makedirs(d, exist_ok=True)
print(os.listdir('/kaggle/input'))

In [ ]:
!touch src/__init__.py

In [ ]:
%%writefile src/config.py
"""Central configuration. Every other module imports from here."""
from dataclasses import dataclass, field
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent


@dataclass
class Paths:
    raw: Path = ROOT / "data" / "raw"
    processed: Path = ROOT / "data" / "processed"
    models: Path = ROOT / "outputs" / "models"
    figures: Path = ROOT / "outputs" / "figures"

    def make_dirs(self) -> None:
        for p in (self.raw, self.processed, self.models, self.figures):
            p.mkdir(parents=True, exist_ok=True)


@dataclass
class Config:
    # --- task ---
    # Start with binary pneumonia; extend this list for multi-label later.
    classes: list = field(default_factory=lambda: ["NORMAL", "PNEUMONIA"])
    task: str = "binary"  # "binary" | "multilabel"

    # --- data ---
    img_size: int = 224
    val_split: float = 0.15
    test_split: float = 0.15
    seed: int = 42
    num_workers: int = 2

    # --- model ---
    backbone: str = "densenet121"  # "densenet121" | "efficientnet_b0"
    pretrained: bool = True
    dropout: float = 0.3

    # --- training ---
    batch_size: int = 32
    epochs: int = 15
    lr: float = 1e-4
    weight_decay: float = 1e-5

    paths: Paths = field(default_factory=Paths)


CFG = Config()


In [ ]:
%%writefile src/data.py
"""Step 3: data scanning, patient-aware 70/15/15 split, transforms, loaders.

Usage (CLI):
    python -m src.data --root /kaggle/input/chest-xray-pneumonia/chest_xray

Design notes
- The original `val/` has only 16 images, so we pool original train+val and
  re-split 70/15/15 (train/val/test). The original `test/` is kept untouched as
  an optional *external* test set (it comes from a different distribution).
- Patient leakage: file names encode patient ids (`person12_bacteria_34.jpeg`,
  `IM-0115-0001.jpeg`, `NORMAL2-IM-0373-0001.jpeg`). We split by patient
  (StratifiedGroupKFold) so one patient never lands in two splits.
  Files with an unparseable name are treated as their own patient.
"""
from __future__ import annotations

import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms as T

from src.config import CFG

EXTS = {".jpeg", ".jpg", ".png"}
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

_PERSON = re.compile(r"person(\d+)_", re.I)
_NORMAL2 = re.compile(r"NORMAL2-IM-(\d+)-", re.I)
_NORMAL = re.compile(r"IM-(\d+)-", re.I)


# ----------------------------------------------------------------- scanning
def find_root(base) -> Path | None:
    """Return the folder that contains train/ and test/."""
    for p in Path(base).rglob("train"):
        if p.is_dir() and (p.parent / "test").exists():
            return p.parent
    return None


def patient_id(path: str | Path) -> str:
    name = Path(path).name
    if m := _PERSON.search(name):
        return f"P{m.group(1)}"
    if m := _NORMAL2.search(name):
        return f"N2-{m.group(1)}"
    if m := _NORMAL.search(name):
        return f"N-{m.group(1)}"
    return f"F-{Path(path).stem}"


def scan_dataset(root, classes=None) -> pd.DataFrame:
    """DataFrame: path, label (str), y (int), orig_split, patient."""
    root = Path(root)
    classes = classes or CFG.classes
    rows = []
    for split in ("train", "val", "test"):
        for y, cls in enumerate(classes):
            d = root / split / cls
            if not d.is_dir():
                continue
            for f in sorted(d.iterdir()):
                if f.suffix.lower() in EXTS:
                    rows.append(dict(path=str(f), label=cls, y=y,
                                     orig_split=split, patient=patient_id(f)))
    if not rows:
        raise FileNotFoundError(f"No images found under {root}")
    return pd.DataFrame(rows)


# ----------------------------------------------------------------- splitting
def make_splits(df: pd.DataFrame, val_frac=None, test_frac=None, seed=None) -> pd.DataFrame:
    """Pool original train+val, split by patient into train/val/test.

    The original test folder is returned with split == "external".
    """
    val_frac = CFG.val_split if val_frac is None else val_frac
    test_frac = CFG.test_split if test_frac is None else test_frac
    seed = CFG.seed if seed is None else seed

    df = df.copy()
    pool = df[df.orig_split.isin(["train", "val"])].copy()

    # 1) carve out test (~test_frac)
    k = max(2, round(1 / test_frac))
    sgkf = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
    rest_idx, test_idx = next(sgkf.split(pool, pool.y, pool.patient))
    test, rest = pool.iloc[test_idx], pool.iloc[rest_idx]

    # 2) carve out val from the rest (~val_frac of the whole pool)
    k = max(2, round((1 - test_frac) / val_frac))
    sgkf = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
    tr_idx, val_idx = next(sgkf.split(rest, rest.y, rest.patient))
    train, val = rest.iloc[tr_idx], rest.iloc[val_idx]

    df["split"] = "external"
    df.loc[train.index, "split"] = "train"
    df.loc[val.index, "split"] = "val"
    df.loc[test.index, "split"] = "test"
    df.loc[df.orig_split == "test", "split"] = "external"
    return df


def check_no_leakage(df: pd.DataFrame) -> None:
    sets = {s: set(df[df.split == s].patient) for s in ("train", "val", "test")}
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = sets[a] & sets[b]
        assert not overlap, f"Patient leakage {a}/{b}: {len(overlap)} patients"


def split_summary(df: pd.DataFrame) -> pd.DataFrame:
    t = df.groupby(["split", "label"]).size().unstack(fill_value=0)
    t["total"] = t.sum(axis=1)
    t["pneumonia_%"] = (100 * t.get("PNEUMONIA", 0) / t["total"]).round(1)
    return t


# ----------------------------------------------------------------- weights
def class_weights(df_train: pd.DataFrame, n_classes: int | None = None) -> torch.Tensor:
    """Inverse-frequency weights, normalised so they average to 1."""
    n_classes = n_classes or len(CFG.classes)
    counts = np.bincount(df_train.y.values, minlength=n_classes).astype(np.float64)
    w = counts.sum() / (n_classes * np.maximum(counts, 1))
    return torch.tensor(w / w.mean(), dtype=torch.float32)


def pos_weight(df_train: pd.DataFrame) -> torch.Tensor:
    """For BCEWithLogitsLoss in the binary setup: n_neg / n_pos."""
    pos = int((df_train.y == 1).sum())
    neg = int((df_train.y == 0).sum())
    return torch.tensor([neg / max(pos, 1)], dtype=torch.float32)


# -------------------------------------------------------------- transforms
def get_transforms(train: bool, size: int | None = None):
    size = size or CFG.img_size
    norm = T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    if train:
        # Mild, anatomy-safe augmentation. No horizontal flip (heart/liver side).
        return T.Compose([
            T.Resize((size, size)),
            T.RandomAffine(degrees=7, translate=(0.05, 0.05), scale=(0.95, 1.05)),
            T.ColorJitter(brightness=0.15, contrast=0.15),
            T.ToTensor(),
            norm,
        ])
    return T.Compose([T.Resize((size, size)), T.ToTensor(), norm])


def denormalize(x: torch.Tensor) -> torch.Tensor:
    """Undo ImageNet normalisation (for plotting / Grad-CAM overlays)."""
    mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(-1, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=x.device).view(-1, 1, 1)
    return (x * std + mean).clamp(0, 1)


# ----------------------------------------------------------------- dataset
class CXRDataset(Dataset):
    """Returns (3xHxW float tensor, label tensor, path)."""

    def __init__(self, df: pd.DataFrame, transform=None):
        self.paths = df.path.tolist()
        self.y = df.y.values
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")  # grayscale -> 3 channels
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.y[i], dtype=torch.long), self.paths[i]


def build_loaders(df: pd.DataFrame, batch_size=None, num_workers=None,
                  balanced_sampler: bool = False) -> dict[str, DataLoader]:
    """Loaders for train / val / test (+ external if present).

    Default is class-weighted loss (see `class_weights`); set
    `balanced_sampler=True` to oversample instead (don't combine both).
    """
    batch_size = batch_size or CFG.batch_size
    num_workers = CFG.num_workers if num_workers is None else num_workers
    pin = torch.cuda.is_available()
    loaders = {}
    for s in ("train", "val", "test", "external"):
        d = df[df.split == s]
        if d.empty:
            continue
        is_train = s == "train"
        ds = CXRDataset(d, get_transforms(train=is_train))
        sampler = None
        if is_train and balanced_sampler:
            w = class_weights(d)[d.y.values].numpy()
            sampler = WeightedRandomSampler(w, num_samples=len(d), replacement=True)
        loaders[s] = DataLoader(ds, batch_size=batch_size,
                                shuffle=is_train and sampler is None,
                                sampler=sampler, num_workers=num_workers,
                                pin_memory=pin, drop_last=False)
    return loaders


# --------------------------------------------------------------------- CLI
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default=None, help="folder containing train/ val/ test/")
    ap.add_argument("--out", default=None, help="where to write splits.csv")
    args = ap.parse_args()

    root = Path(args.root) if args.root else find_root(CFG.paths.raw) or find_root("/kaggle/input")
    if root is None:
        raise SystemExit("Dataset not found. Pass --root.")
    CFG.paths.make_dirs()
    out = Path(args.out) if args.out else CFG.paths.processed / "splits.csv"

    df = make_splits(scan_dataset(root))
    check_no_leakage(df)
    df.to_csv(out, index=False)

    print(f"root: {root}\nsaved: {out}\n")
    print(split_summary(df))
    print("\nunique patients per split:")
    print(df.groupby("split").patient.nunique())
    print("\nclass weights (train):", class_weights(df[df.split == 'train']).tolist())
    print("pos_weight  (train):", pos_weight(df[df.split == 'train']).tolist())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/model.py
"""Step 4: model factory (DenseNet121 / EfficientNet-B0) + helpers.

Binary task  -> 1 logit, use BCEWithLogitsLoss(pos_weight=data.pos_weight(...)).
Multi-label  -> len(classes) logits, same loss (pos_weight per class if needed).
"""
from __future__ import annotations

import warnings

import torch
import torch.nn as nn
from torchvision import models

from src.config import CFG


def n_outputs(classes=None, task=None) -> int:
    classes = classes or CFG.classes
    task = task or CFG.task
    return 1 if task == "binary" else len(classes)


def build_model(backbone: str | None = None, pretrained: bool | None = None,
                dropout: float | None = None, num_outputs: int | None = None) -> nn.Module:
    backbone = backbone or CFG.backbone
    pretrained = CFG.pretrained if pretrained is None else pretrained
    dropout = CFG.dropout if dropout is None else dropout
    num_outputs = num_outputs or n_outputs()

    if backbone == "densenet121":
        w = models.DenseNet121_Weights.DEFAULT if pretrained else None
        m = _load(models.densenet121, w)
        m.classifier = nn.Sequential(nn.Dropout(dropout),
                                     nn.Linear(m.classifier.in_features, num_outputs))
    elif backbone == "efficientnet_b0":
        w = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        m = _load(models.efficientnet_b0, w)
        m.classifier = nn.Sequential(nn.Dropout(dropout),
                                     nn.Linear(m.classifier[1].in_features, num_outputs))
    else:
        raise ValueError(f"Unknown backbone: {backbone}")
    return m


def _load(fn, weights):
    """Load with pretrained weights; fall back to random init (loudly) if the
    download fails, e.g. Kaggle notebook with Internet turned off."""
    try:
        return fn(weights=weights)
    except Exception as e:  # noqa: BLE001
        warnings.warn(
            f"\n!!! Could not load pretrained weights ({type(e).__name__}). "
            "Using RANDOM init. Turn Internet ON in the notebook settings.\n")
        return fn(weights=None)


def gradcam_target_layer(model: nn.Module, backbone: str | None = None):
    """Last conv feature block, used by Grad-CAM in step 6."""
    backbone = backbone or CFG.backbone
    if backbone in ("densenet121", "efficientnet_b0"):
        return model.features[-1]
    raise ValueError(f"Unknown backbone: {backbone}")


def set_backbone_trainable(model: nn.Module, trainable: bool) -> None:
    """Freeze/unfreeze everything except the classifier head."""
    for name, p in model.named_parameters():
        if not name.startswith("classifier"):
            p.requires_grad = trainable


def count_params(model: nn.Module) -> tuple[int, int]:
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, train


def logits_to_prob(logits: torch.Tensor) -> torch.Tensor:
    return torch.sigmoid(logits)


if __name__ == "__main__":
    m = build_model()
    x = torch.randn(2, 3, CFG.img_size, CFG.img_size)
    out = m(x)
    print("backbone:", CFG.backbone, "| output shape:", tuple(out.shape))
    print("params (total, trainable):", count_params(m))
    print("gradcam layer:", type(gradcam_target_layer(m)).__name__)


In [ ]:
%%writefile src/evaluate.py
"""Step 5b: evaluation (metrics, threshold selection, ROC / confusion plots)."""
from __future__ import annotations

import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score, roc_curve

from src.config import CFG


@torch.no_grad()
def predict(model, loader, device):
    """Return (y_true, prob_pneumonia) as numpy arrays."""
    model.eval()
    ys, ps = [], []
    for x, y, _ in loader:
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            logits = model(x.to(device, non_blocking=True)).squeeze(1)
        ps.append(torch.sigmoid(logits.float()).cpu())
        ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()


def safe_auc(y, p) -> float:
    try:
        return float(roc_auc_score(y, p))
    except ValueError:  # only one class present
        return float("nan")


def best_threshold(y, p) -> float:
    """Youden's J (sensitivity + specificity - 1). Choose it on VAL only."""
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[int(np.argmax(tpr - fpr))])


def compute_metrics(y, p, thr: float = 0.5) -> dict:
    y = np.asarray(y).astype(int)
    pred = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    prec = tp / max(tp + fp, 1)
    f1 = 2 * prec * sens / max(prec + sens, 1e-9)
    return dict(
        threshold=round(float(thr), 4),
        accuracy=round(float((tp + tn) / len(y)), 4),
        sensitivity=round(float(sens), 4),
        specificity=round(float(spec), 4),
        precision=round(float(prec), 4),
        f1=round(float(f1), 4),
        auroc=round(safe_auc(y, p), 4),
        auprc=round(float(average_precision_score(y, p)), 4),
        tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
    )


def evaluate_all(model, loaders: dict, device, thr: float, out_dir=None) -> pd.DataFrame:
    """Evaluate val/test/external, save metrics.json + roc.png + confusion.png."""
    out_dir = Path(out_dir or CFG.paths.figures)
    out_dir.mkdir(parents=True, exist_ok=True)
    rows, preds = {}, {}
    for name in ("val", "test", "external"):
        if name not in loaders:
            continue
        y, p = predict(model, loaders[name], device)
        preds[name] = (y, p)
        rows[name] = compute_metrics(y, p, thr)
    table = pd.DataFrame(rows).T
    (out_dir / "metrics.json").write_text(json.dumps(rows, indent=2))
    table.to_csv(out_dir / "metrics.csv")

    # ROC curves
    fig, ax = plt.subplots(figsize=(5, 5))
    for name, (y, p) in preds.items():
        fpr, tpr, _ = roc_curve(y, p)
        ax.plot(fpr, tpr, label=f"{name} (AUC {rows[name]['auroc']:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set_xlabel("1 - specificity"); ax.set_ylabel("sensitivity"); ax.legend(); ax.set_title("ROC")
    fig.tight_layout(); fig.savefig(out_dir / "roc.png", dpi=150); plt.close(fig)

    # Confusion matrices
    fig, axes = plt.subplots(1, len(rows), figsize=(4 * len(rows), 3.6))
    axes = np.atleast_1d(axes)
    for ax, (name, m) in zip(axes, rows.items()):
        cm = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
        ax.imshow(cm, cmap="Blues")
        for (i, j), v in np.ndenumerate(cm):
            ax.text(j, i, v, ha="center", va="center", fontsize=13)
        ax.set_xticks([0, 1]); ax.set_xticklabels(CFG.classes, fontsize=8)
        ax.set_yticks([0, 1]); ax.set_yticklabels(CFG.classes, fontsize=8)
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{name} (thr {m['threshold']})")
    fig.tight_layout(); fig.savefig(out_dir / "confusion.png", dpi=150); plt.close(fig)
    return table


In [ ]:
%%writefile src/train.py
"""Step 5a: training loop.

    python -m src.train --smoke      # quick pipeline check (tiny subset, 1 epoch)
    python -m src.train              # full training (needs GPU)
"""
from __future__ import annotations

import argparse
import random
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from src.config import CFG
from src.data import build_loaders, pos_weight
from src.evaluate import best_threshold, evaluate_all, predict, safe_auc
from src.model import build_model


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def run_epoch(model, loader, criterion, device, optimizer=None, scaler=None):
    train = optimizer is not None
    model.train(train)
    use_amp = device.type == "cuda"
    total, n = 0.0, 0
    ys, ps = [], []
    with torch.set_grad_enabled(train):
        for x, y, _ in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device).float()
            with torch.autocast(device_type=device.type, enabled=use_amp):
                logits = model(x).squeeze(1)
            loss = criterion(logits.float(), y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total += loss.item() * len(y)
            n += len(y)
            ys.append(y.cpu())
            ps.append(torch.sigmoid(logits.detach().float()).cpu())
    y = torch.cat(ys).numpy()
    p = torch.cat(ps).numpy()
    return total / n, safe_auc(y, p), float(((p >= 0.5) == y).mean())


def plot_history(hist: pd.DataFrame, path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    axes[0].plot(hist.epoch, hist.train_loss, label="train")
    axes[0].plot(hist.epoch, hist.val_loss, label="val")
    axes[0].set_title("loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[1].plot(hist.epoch, hist.train_auc, label="train")
    axes[1].plot(hist.epoch, hist.val_auc, label="val")
    axes[1].set_title("AUROC"); axes[1].set_xlabel("epoch"); axes[1].legend()
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=CFG.epochs)
    ap.add_argument("--batch_size", type=int, default=CFG.batch_size)
    ap.add_argument("--lr", type=float, default=CFG.lr)
    ap.add_argument("--patience", type=int, default=4)
    ap.add_argument("--splits", default=str(CFG.paths.processed / "splits.csv"))
    ap.add_argument("--smoke", action="store_true", help="tiny subset, 1 epoch")
    args = ap.parse_args()

    seed_everything(CFG.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)
    if device.type == "cpu" and not args.smoke:
        print("!!! No GPU. Turn on GPU in Session options (Accelerator), or use --smoke.")

    df = pd.read_csv(args.splits)
    if args.smoke:
        df = df.groupby(["split", "label"]).head(16)
        args.epochs = 1
    loaders = build_loaders(df, batch_size=args.batch_size)
    train_df = df[df.split == "train"]

    CFG.paths.make_dirs()
    ckpt = CFG.paths.models / ("smoke.pt" if args.smoke else "best.pt")

    model = build_model().to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight(train_df).to(device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    hist, best_auc, bad = [], -1.0, 0
    for epoch in range(1, args.epochs + 1):
        t0 = time.time()
        tr_loss, tr_auc, tr_acc = run_epoch(model, loaders["train"], criterion, device, optimizer, scaler)
        va_loss, va_auc, va_acc = run_epoch(model, loaders["val"], criterion, device)
        scheduler.step()
        hist.append(dict(epoch=epoch, train_loss=tr_loss, train_auc=tr_auc, train_acc=tr_acc,
                         val_loss=va_loss, val_auc=va_auc, val_acc=va_acc))
        improved = np.nan_to_num(va_auc, nan=0.0) > best_auc
        if improved:
            best_auc, bad = float(np.nan_to_num(va_auc, nan=0.0)), 0
            torch.save({"model": model.state_dict(), "backbone": CFG.backbone,
                        "epoch": epoch, "val_auc": va_auc}, ckpt)
        else:
            bad += 1
        print(f"epoch {epoch:02d}/{args.epochs} | train loss {tr_loss:.4f} auc {tr_auc:.4f} | "
              f"val loss {va_loss:.4f} auc {va_auc:.4f} acc {va_acc:.4f} | "
              f"{time.time() - t0:.0f}s {'*best*' if improved else ''}")
        if bad >= args.patience:
            print(f"early stop (no val AUROC improvement for {args.patience} epochs)")
            break

    hist = pd.DataFrame(hist)
    hist.to_csv(CFG.paths.figures / "history.csv", index=False)
    plot_history(hist, CFG.paths.figures / "history.png")

    # ---- final evaluation with the best checkpoint ----
    model.load_state_dict(torch.load(ckpt, map_location=device)["model"])
    yv, pv = predict(model, loaders["val"], device)
    thr = best_threshold(yv, pv)  # chosen on VAL only, then applied to test/external
    table = evaluate_all(model, loaders, device, thr)
    print(f"\nthreshold from val (Youden J): {thr:.4f}\n")
    print(table.to_string())
    print("\nsaved:", ckpt, "and figures in", CFG.paths.figures)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/gradcam.py
"""Step 6: Grad-CAM (self-contained, no extra dependency).

    python -m src.gradcam            # galleries for test + external -> outputs/figures/

Binary model: the target is the single "PNEUMONIA" logit, so the heatmap shows
which regions pushed the prediction toward pneumonia.
"""
from __future__ import annotations

import argparse
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from src.config import CFG
from src.data import CXRDataset, denormalize, get_transforms
from src.evaluate import predict
from src.model import build_model


class GradCAM:
    """Grad-CAM on the last feature maps of DenseNet121 / EfficientNet-B0.

    We re-run the forward pass by hand (features -> pool -> classifier) so the
    feature maps are a normal tensor we can call retain_grad() on. This avoids
    the in-place ReLU / backward-hook problems of DenseNet.
    """

    def __init__(self, model, backbone: str | None = None):
        self.model = model
        self.backbone = backbone or CFG.backbone

    def _forward(self, x):
        f = self.model.features(x)
        a = F.relu(f) if self.backbone == "densenet121" else f
        a.retain_grad()
        pooled = F.adaptive_avg_pool2d(a, 1).flatten(1)
        return self.model.classifier(pooled), a

    def __call__(self, x):
        """x: (B,3,H,W) normalised. Returns cam (B,H,W) in [0,1], prob (B,)."""
        self.model.eval()
        with torch.enable_grad():
            self.model.zero_grad(set_to_none=True)
            logits, a = self._forward(x)
            logits.squeeze(1).sum().backward()
        w = a.grad.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * a).sum(dim=1, keepdim=True)).detach()
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
        mn = cam.amin(dim=(1, 2), keepdim=True)
        mx = cam.amax(dim=(1, 2), keepdim=True)
        cam = (cam - mn) / (mx - mn + 1e-8)
        return cam.cpu(), torch.sigmoid(logits.detach()).squeeze(1).cpu()


def overlay(img: torch.Tensor, cam: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """img: (3,H,W) normalised tensor, cam: (H,W) in [0,1] -> (H,W,3) float image."""
    base = denormalize(img.cpu()).permute(1, 2, 0).numpy()
    heat = plt.get_cmap("jet")(cam)[..., :3]
    return np.clip((1 - alpha) * base + alpha * heat, 0, 1)


def explain_pil(model, pil_img, device, backbone: str | None = None):
    """For the demo app: PIL image -> (probability of pneumonia, overlay array)."""
    x = get_transforms(train=False)(pil_img.convert("RGB"))[None].to(device)
    cam, prob = GradCAM(model, backbone)(x)
    return float(prob[0]), overlay(x[0], cam[0].numpy())


def select_cases(y, p, thr, n, seed=CFG.seed):
    pred = (p >= thr).astype(int)
    rng = np.random.default_rng(seed)
    out = {}
    for name, (yt, yp) in {"TP": (1, 1), "TN": (0, 0), "FP": (0, 1), "FN": (1, 0)}.items():
        idx = np.where((y == yt) & (pred == yp))[0]
        out[name] = list(rng.choice(idx, size=min(n, len(idx)), replace=False)) if len(idx) else []
    return out


def gallery(model, df_split: pd.DataFrame, device, thr: float, n: int, out_path, title: str):
    ds = CXRDataset(df_split, get_transforms(train=False))
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=CFG.num_workers)
    y, p = predict(model, loader, device)  # same order as df_split
    cases = select_cases(y, p, thr, n)
    gc = GradCAM(model)

    fig, axes = plt.subplots(4, n, figsize=(2.6 * n, 2.8 * 4), squeeze=False)
    for r, (name, idxs) in enumerate(cases.items()):
        for c in range(n):
            ax = axes[r][c]
            ax.axis("off")
            if c >= len(idxs):
                continue
            x, _, _ = ds[int(idxs[c])]
            cam, prob = gc(x[None].to(device))
            ax.imshow(overlay(x, cam[0].numpy()))
            ax.set_title(f"{name}  p={float(prob[0]):.2f}", fontsize=9)
    fig.suptitle(f"{title}  (thr={thr:.3f}; TP/TN correct, FP/FN errors)", fontsize=11)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print("saved", out_path, {k: len(v) for k, v in cases.items()})


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=5, help="examples per category")
    ap.add_argument("--splits", default=str(CFG.paths.processed / "splits.csv"))
    ap.add_argument("--ckpt", default=str(CFG.paths.models / "best.pt"))
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(pretrained=False).to(device)
    model.load_state_dict(torch.load(args.ckpt, map_location=device)["model"])

    thr = 0.5
    mpath = CFG.paths.figures / "metrics.json"
    if mpath.exists():
        thr = json.loads(mpath.read_text())["val"]["threshold"]

    df = pd.read_csv(args.splits)
    CFG.paths.make_dirs()
    for split in ("test", "external"):
        gallery(model, df[df.split == split], device, thr, args.n,
                CFG.paths.figures / f"gradcam_{split}.png", f"Grad-CAM: {split}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/shortcut_check.py
"""Step 6b: does the model rely on image corners / borders (markers, text, edges)?

Re-evaluates val/test/external with the same threshold while hiding regions:
  original | corners masked (laterality marker / text) | outer border masked
A big drop when a region is hidden = the model uses it (possible shortcut).

    python -m src.shortcut_check
"""
from __future__ import annotations

import json

import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.config import CFG
from src.data import CXRDataset, get_transforms
from src.evaluate import compute_metrics, predict
from src.model import build_model


def mask_corners(x, f=0.22):
    h, w = int(x.shape[-2] * f), int(x.shape[-1] * f)
    x = x.clone()
    x[:, :h, :w] = 0      # top-left (usually the "R" marker)
    x[:, :h, -w:] = 0     # top-right
    x[:, -h:, :w] = 0     # bottom-left
    x[:, -h:, -w:] = 0    # bottom-right (usually text)
    return x


def mask_border(x, f=0.12):
    h, w = int(x.shape[-2] * f), int(x.shape[-1] * f)
    out = torch.zeros_like(x)
    out[:, h:-h, w:-w] = x[:, h:-h, w:-w]
    return out


VARIANTS = {"original": lambda x: x, "corners_masked": mask_corners, "border_masked": mask_border}


class MaskedDataset(CXRDataset):
    def __init__(self, df, fn):
        super().__init__(df, get_transforms(train=False))
        self.fn = fn

    def __getitem__(self, i):
        x, y, p = super().__getitem__(i)
        return self.fn(x), y, p


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(pretrained=False).to(device)
    model.load_state_dict(torch.load(CFG.paths.models / "best.pt", map_location=device)["model"])
    thr = json.loads((CFG.paths.figures / "metrics.json").read_text())["val"]["threshold"]
    df = pd.read_csv(CFG.paths.processed / "splits.csv")

    rows = []
    for split in ("test", "external"):
        d = df[df.split == split]
        for name, fn in VARIANTS.items():
            loader = DataLoader(MaskedDataset(d, fn), batch_size=64, shuffle=False,
                                num_workers=CFG.num_workers)
            y, p = predict(model, loader, device)
            m = compute_metrics(y, p, thr)
            rows.append(dict(split=split, variant=name, accuracy=m["accuracy"],
                             sensitivity=m["sensitivity"], specificity=m["specificity"],
                             auroc=m["auroc"], fp=m["fp"], fn=m["fn"]))
    out = pd.DataFrame(rows)
    out.to_csv(CFG.paths.figures / "shortcut_check.csv", index=False)
    print(f"threshold (from val): {thr:.4f}\n")
    print(out.to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile app/demo.py
"""Step 7: Gradio demo - upload a chest X-ray, get a probability + Grad-CAM heatmap.

    pip install gradio
    python app/demo.py --ckpt outputs/models/best.pt [--share]

Research demo only. NOT a medical device.
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))

import gradio as gr  # noqa: E402
import numpy as np  # noqa: E402
import torch  # noqa: E402

from src.config import CFG  # noqa: E402
from src.gradcam import explain_pil  # noqa: E402
from src.model import build_model  # noqa: E402

DISCLAIMER = (
    "**Research demo - not a medical device.** Trained on pediatric chest X-rays "
    "(Kermany et al., Guangzhou) only. On the held-out external set specificity was "
    "~72% at the chosen threshold, so false alarms are common. "
    "**Adult or non-chest images are outside the training distribution, so the output is not meaningful.** "
    "The heatmap shows what raised the pneumonia score (it is weak/meaningless when the score is low); "
    "it is not proof of the cause."
)


def load_model(ckpt, device):
    model = build_model(pretrained=False).to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device)["model"])
    return model.eval()


def default_threshold() -> float:
    p = CFG.paths.figures / "metrics.json"
    return json.loads(p.read_text())["val"]["threshold"] if p.exists() else 0.5


def build_app(model, device, thr):
    def analyze(img, threshold):
        if img is None:
            return None, "Please upload an image."
        prob, cam = explain_pil(model, img, device)
        label = "Pneumonia pattern suspected" if prob >= threshold else "No pneumonia pattern found"
        text = f"{label}\nP(pneumonia) = {prob:.3f}   (threshold {threshold:.2f})"
        return (cam * 255).astype(np.uint8), text

    with gr.Blocks(title="Explainable chest X-ray pneumonia detection") as demo:
        gr.Markdown("# Explainable Chest X-ray Pneumonia Detection")
        gr.Markdown(DISCLAIMER)
        with gr.Row():
            inp = gr.Image(type="pil", label="Chest X-ray")
            out = gr.Image(label="Grad-CAM overlay")
        slider = gr.Slider(0.05, 0.95, value=thr, step=0.01, label="Decision threshold")
        result = gr.Textbox(label="Result", lines=2)
        gr.Button("Analyze").click(analyze, [inp, slider], [out, result])
    return demo


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", default=str(CFG.paths.models / "best.pt"))
    ap.add_argument("--thr", type=float, default=None)
    ap.add_argument("--share", action="store_true", help="public temporary link (e.g. on Kaggle)")
    args = ap.parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = load_model(args.ckpt, device)
    thr = args.thr if args.thr is not None else default_threshold()
    build_app(model, device, thr).launch(share=args.share)


if __name__ == "__main__":
    main()


## 1) Splits

In [ ]:
!python -m src.data

## 2) Smoke test, then full training

In [ ]:
!python -m src.train --smoke

In [ ]:
!python -m src.train

## 3) Explainability and shortcut check

In [ ]:
!python -m src.gradcam

In [ ]:
!python -m src.shortcut_check

In [ ]:
from IPython.display import Image, display
for n in ('gradcam_test','gradcam_external','roc','confusion','history'):
    display(Image(f'/kaggle/working/outputs/figures/{n}.png'))

## 4) Demo (optional)
Run `!pip -q install gradio` then `!python app/demo.py`. It prints a temporary public link; stop the cell when done.